# Ambient Budget-Validation Agent: Architectural Guide & Implementation Walkthrough

This notebook provides a detailed explanation of the **Ambient Budget-Validation Agent**, an event-driven system built using the **Google Agent Development Kit (ADK) 2.0 Graph Workflow API**.

---

## ⚡ Running on Kaggle / Google Colab

If you are running this notebook inside **Kaggle** or **Google Colab** instead of locally, run the cell below to clone the project repository, set the working directory, and install the required ADK and Google Cloud libraries.

### 🔑 Google Cloud Authentication via Kaggle Secrets:
To retrieve your GCP Project ID safely from Kaggle's Secrets store, this notebook accesses the `GOOGLE_CLOUD_PROJECT` secret key.
1. Go to **Add-ons** -> **Secrets** in the Kaggle notebook menu.
2. Add a secret with Label `GOOGLE_CLOUD_PROJECT` and Value set to your Google Cloud Project ID.
3. Grant the notebook permission to access the secret.

In [ ]:
# 1. Clone the project repository to access local source modules
import os
if not os.path.exists("Ambient-Budget-Validation-Agent"):
    !git clone https://github.com/Divik-MLEngineer/Ambient-Budget-Validation-Agent.git

%cd Ambient-Budget-Validation-Agent

# 2. Force upgrade dependencies to ensure 2.x versions are installed
!pip install --upgrade --quiet google-adk fastapi uvicorn httpx google-cloud-aiplatform google-auth pydantic

# 3. Force reload the 'google' namespace cache so upgraded packages are visible immediately
import sys
for key in list(sys.modules.keys()):
    if key == "google" or key.startswith("google."):
        sys.modules.pop(key, None)

# 4. Load GOOGLE_CLOUD_PROJECT from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    os.environ["GOOGLE_CLOUD_PROJECT"] = user_secrets.get_secret("GOOGLE_CLOUD_PROJECT")
    if not os.environ.get("GOOGLE_CLOUD_PROJECT"):
        raise ValueError("Secret key GOOGLE_CLOUD_PROJECT is empty. Please check your Kaggle Secrets config.")
    print("Successfully set GOOGLE_CLOUD_PROJECT from Kaggle Secrets:", os.environ["GOOGLE_CLOUD_PROJECT"])
except Exception as e:
    # Fallback to prompt user to set it manually
    if not os.environ.get("GOOGLE_CLOUD_PROJECT"):
        os.environ["GOOGLE_CLOUD_PROJECT"] = "YOUR_PROJECT_ID"  # Replace with your GCP project ID
        print("Kaggle Secrets client not available. Fallback default applied. Please replace 'YOUR_PROJECT_ID' with your project ID.")
    else:
        print("Using existing project environment context:", os.environ["GOOGLE_CLOUD_PROJECT"])

print("Environment configuration complete!")

## 1. Project Overview & Features
The agent's role is to automatically ingest purchase requests arriving via Pub/Sub, assess their compliance against financial budgets, enforce security controls, run AI risk audits when necessary, and handle Human-in-the-Loop (HIL) approvals.

### Key Features:
1. **Graph Workflow routing**: Within-budget requests are approved instantly; over-budget requests trigger an AI audit and prompt human reviews.
2. **PII Redaction**: Intercepts descriptions to strip Social Security Numbers (SSN) and Credit Card details.
3. **Prompt Injection Defense**: Bypasses downstream LLM blocks if malicious instructions are detected in the description, routing the event directly to human reviewers as a security flag.
4. **Ambient Servicing**: Exposes a FastAPI endpoint on port 8080 optimized for Pub/Sub push subscriptions.
5. **Quality Flywheel & Evals**: Uses custom LLM-as-judge metrics (`routing_correctness` and `security_containment`) to grade workflow behavior.

## 2. Pydantic Models & Data Schemas
The workflow is strictly typed using Pydantic models. This ensures correct contracts between nodes.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

class PurchaseRequest(BaseModel):
    """Contract representing an incoming purchase request."""
    request_id: str
    department: str
    project: str
    requested_amount: float
    available_budget: float
    requester: str
    description: str
    date: str

class RiskAssessment(BaseModel):
    """Schema for AI risk auditor outputs."""
    risk_level: str = Field(description="Risk level: low, medium, or high")
    risk_factors: List[str] = Field(description="List of budget compliance risk factors")
    alert_raised: bool = Field(description="True if an alert is raised due to high risk")
    justification: str = Field(description="Detailed reasoning for the risk assessment")
    security_event: Optional[bool] = Field(default=False, description="True if prompt injection was detected")

class ValidationResult(BaseModel):
    """Final outcome contract representing workflow results."""
    approved: bool
    status: str  # auto_approved, approved, or rejected
    reason: str

## 3. Workflow Graph Design
The workflow graph wires together functional nodes. Below is the conceptual architecture of the graph:

```
                      [Incoming Purchase Request]
                                   │
                                   ▼
                            [parse_input]
                                   │
                                   ▼
                           [validate_budget]
                            /             \
                (Within Budget)         (Over Budget)
                      /                     \
                     ▼                       ▼
              [record_outcome]      [security_checkpoint]
                     │                /               \
                     │        (Security Event)      (Clean)
                     │             /                     \
                     │            ▼                       ▼
                     │     [human_approval]         [llm_reviewer]
                     │            │                       │
                     │            │                       ▼
                     │            │                [human_approval]
                     │            │                       │
                     └────────────┼───────────────────────┘
                                  │
                                  ▼
                           [record_outcome]
```

## 4. Key Node Implementations

### The Security Checkpoint Node
The security checkpoint scrubs personal PII from descriptions and checks for prompt injections.

In [ ]:
import re
from google.adk.events import Event

def clean_pii(text: str) -> str:
    """Scrubs SSN and Credit Card numbers from text."""
    # Match SSN format (e.g. 000-00-0000 or 9 digits)
    ssn_pattern = re.compile(r'\b\d{3}-\d{2}-\d{4}\b|\b\d{9}\b')
    text = ssn_pattern.sub('[REDACTED_SSN]', text)
    
    # Match Credit Card format (e.g. 1111-2222-3333-4444 or 16 digits)
    cc_pattern = re.compile(r'\b\d{4}-\d{4}-\d{4}-\d{4}\b|\b\d{16}\b')
    text = cc_pattern.sub('[REDACTED_CC]', text)
    return text

def detect_prompt_injection(text: str) -> bool:
    """Detects attempts to bypass workflow validation logic."""
    triggers = [
        "ignore previous",
        "bypass all",
        "always approve",
        "system override",
        "ignore instructions"
    ]
    text_lower = text.lower()
    return any(t in text_lower for t in triggers)

### Human in the Loop (HIL) Resumption Node
The HIL node pauses the graph, yields a `RequestInput` event, and waits for a manager's resume payload.

In [ ]:
from google.adk.agents.context import Context
from google.adk.events import RequestInput
from google.adk.workflow import node

@node(name="human_approval", rerun_on_resume=True)
async def human_approval_node(ctx: Context, node_input: RiskAssessment):
    """HIL node that prompts a user and resumes on response."""
    if not ctx.resume_inputs or "decision" not in ctx.resume_inputs:
        purchase_request_dict = ctx.state.get("purchase_request", {})
        req_id = purchase_request_dict.get("request_id", "unknown")
        amount = purchase_request_dict.get("requested_amount", 0.0)
        budget = purchase_request_dict.get("available_budget", 0.0)
        
        msg = (
            f"WARNING: Purchase Request {req_id} exceeds budget! "
            f"Requested: ${amount}, Available: ${budget}. "
            f"LLM Audit -> Risk Level: {node_input.risk_level.upper()}. "
            f"Justification: {node_input.justification}. "
            "Approve or reject?"
        )
        # Halt execution and yield interrupt
        yield RequestInput(interrupt_id="decision", message=msg)
        return

    # Workflow resumes here once response is provided
    decision = ctx.resume_inputs["decision"]
    if isinstance(decision, dict):
        decision_val = decision.get("decision", decision.get("response", str(decision)))
    else:
        decision_val = decision
        
    approved = str(decision_val).lower() in ("approve", "approved", "yes", "true")
    status = "approved" if approved else "rejected"
    reason = f"Human reviewer {status} the request. Comment: {decision_val}"
    
    yield Event(output=ValidationResult(approved=approved, status=status, reason=reason))

## 5. Live Use Case Execution (Direct in Notebook)
The code cells below run the actual workflow graph directly inside your Kaggle notebook environment so you can view outputs in real-time.

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import json
from budget_validation_agent.agent import root_agent

# Create local runner instance
session_service = InMemorySessionService()
runner = Runner(
    agent=root_agent,
    session_service=session_service,
    app_name="budget_validation_agent",
)

def print_event(event):
    author = event.author or "system"
    if event.output:
        print(f"[{author.upper()} Node Output]:", event.output)
    elif event.content:
        # Check if the content contains standard text parts
        parts = getattr(event.content, 'parts', [])
        for part in parts:
            if hasattr(part, 'text') and part.text:
                print(f"[{author.upper()} Message]:", part.text)
            elif hasattr(part, 'function_call') and part.function_call:
                print(f"[{author.upper()} FunctionCall]:", part.function_call.name, "(ID:", part.function_call.id, ")")
            elif hasattr(part, 'function_response') and part.function_response:
                print(f"[{author.upper()} FunctionResponse]:", part.function_response.name)
    else:
        if event.long_running_tool_ids:
            print(f"[SYSTEM] Interrupt paused on IDs: {list(event.long_running_tool_ids)}")

### Scenario 1: Auto-Approved Request (Within Budget)
A purchase request under budget that approves instantly.

In [ ]:
within_budget_payload = {
    "request_id": "PR-KAGGL-001",
    "department": "Engineering",
    "project": "AI Platform",
    "requested_amount": 450.00,
    "available_budget": 1000.00,
    "requester": "user@example.com",
    "description": "Development tools subscription",
    "date": "2026-07-07"
}

print("=== STARTING WITHIN-BUDGET SCENARIO ===")

# Create the session on the session service first to avoid SessionNotFoundError
session = await session_service.create_session(
    app_name="budget_validation_agent",
    user_id="default-user"
)

user_message = types.Content(
    role="user",
    parts=[types.Part.from_text(text=json.dumps(within_budget_payload))]
)

async for event in runner.run_async(
    user_id="default-user",
    session_id=session.id,
    new_message=user_message
):
    print_event(event)
print("=== FINISHED ===")

### Scenario 2: Over-Budget Request (Triggers HIL & Resume)
A purchase request exceeding budget that pauses for human override, which is then programmatically resumed.

In [ ]:
over_budget_payload = {
    "request_id": "PR-KAGGL-002",
    "department": "Marketing",
    "project": "Campaign",
    "requested_amount": 2500.00,
    "available_budget": 1000.00,
    "requester": "user@example.com",
    "description": "Campaign ad budget",
    "date": "2026-07-07"
}

print("=== STARTING OVER-BUDGET WORKFLOW ===")

# Create the session on the session service first to avoid SessionNotFoundError
session = await session_service.create_session(
    app_name="budget_validation_agent",
    user_id="default-user"
)

user_message = types.Content(
    role="user",
    parts=[types.Part.from_text(text=json.dumps(over_budget_payload))]
)

# 1. Run until pause
is_interrupted = False
async for event in runner.run_async(
    user_id="default-user",
    session_id=session.id,
    new_message=user_message
):
    print_event(event)
    if event.long_running_tool_ids:
        is_interrupted = True

# 2. If interrupted, simulate manager approval response
if is_interrupted:
    print(
        "\n[SIMULATION] Manager reviews details and decides to APPROVE. Resuming session..."
    )
    resume_message = types.Content(
        role="user",
        parts=[
            types.Part(
                function_response=types.FunctionResponse(
                    name="adk_request_input",
                    id="decision",
                    response={"decision": "approve"},
                )
            )
        ],
    )

    async for event in runner.run_async(
        user_id="default-user",
        session_id=session.id,
        new_message=resume_message,
    ):
        print_event(event)
print("=== FINISHED ===")

## 6. Remote Testing & Deployed Reasoning Engine
Once deployed to Vertex AI Agent Runtime, the agent is queried using OAuth authentication tokens. Below is an example of querying the remote engine using Python.

In [ ]:
import httpx
import google.auth
import google.auth.transport.requests
import json

def query_remote_agent(project_id: str, location: str, engine_id: str, payload: dict):
    """Queries the live reasoning engine endpoint."""
    credentials, _ = google.auth.default()
    auth_request = google.auth.transport.requests.Request()
    credentials.refresh(auth_request)
    token = credentials.token

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

    url = f"https://{location}-aiplatform.googleapis.com/v1/projects/{project_id}/locations/{location}/reasoningEngines/{engine_id}:streamQuery"
    
    input_payload = {
        "class_method": "async_stream_query",
        "input": {
            "message": {
                "role": "user",
                "parts": [{"text": json.dumps(payload)}]
            },
            "user_id": "sample-user"
        }
    }
    # response = httpx.post(url, headers=headers, json=input_payload)
    # print(response.text)
    pass

## 7. Evaluation Loop & Grading
Quality metrics are evaluated on traces using LLM-as-judge grading. Two custom metrics are declared in `tests/eval/eval_config.yaml`:

1. **`routing_correctness`**: Confirms that within-budget requests validate instantly, while over-budget requests stop at HIL.
2. **`security_containment`**: Checks that SSNs and Credit Cards are redacted, and prompt injections bypass the LLM risk auditor.

To execute local evaluations and grade the traces:
```bash
# 1. Run eval cases and save traces
make generate-traces

# 2. Grade traces against LLM judge
make grade
```